# Experiment Tracking with MLflow

Complete walkthrough: log runs manually, compare with autologging, reload a model,
and register the best one. Install deps first: `pip install -r ../requirements.txt`.

In a terminal at this `notebooks/` folder, run the UI against the same backend:
```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

## 1. Setup: point MLflow at a local SQLite backend (enables the Model Registry)

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("iris-baseline")
print("tracking URI:", mlflow.get_tracking_uri())

## 2. Manual logging: params, metrics and the model artifact across several runs

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for C in [0.01, 0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f"logreg-C={C}"):
        model = LogisticRegression(C=C, max_iter=500).fit(X_train, y_train)
        preds = model.predict(X_test)

        mlflow.log_param("C", C)
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_metric("accuracy", accuracy_score(y_test, preds))
        mlflow.log_metric("f1_macro", f1_score(y_test, preds, average="macro"))
        mlflow.sklearn.log_model(model, name="model")

print("logged 4 runs — refresh the MLflow UI to compare them")

## 3. Autologging: same experiment, near-zero logging code

In [ ]:
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="rf-autologged"):
    rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=0)
    rf.fit(X_train, y_train)
    # autolog captured params, training metrics and the model automatically;
    # still worth logging the metric YOU care about on YOUR test set:
    mlflow.log_metric("accuracy", rf.score(X_test, y_test))

mlflow.sklearn.autolog(disable=True)

## 4. Compare runs programmatically and reload the best model

In [ ]:
runs = mlflow.search_runs(order_by=["metrics.accuracy DESC"])
cols = ["run_id", "tags.mlflow.runName", "params.C", "metrics.accuracy"]
display(runs[[c for c in cols if c in runs.columns]].head())

best = runs.iloc[0]
model_uri = f"runs:/{best.run_id}/model"
loaded = mlflow.pyfunc.load_model(model_uri)
print("reloaded best model, sample predictions:", loaded.predict(X_test[:5]))

## 5. Register the best run in the Model Registry and assign an alias

In [ ]:
from mlflow import MlflowClient

result = mlflow.register_model(model_uri, "iris-classifier")
client = MlflowClient()
client.set_registered_model_alias("iris-classifier", "staging", result.version)

# Serving code would now load it WITHOUT knowing any run id:
staged = mlflow.pyfunc.load_model("models:/iris-classifier@staging")
print(f"version {result.version} is @staging; prediction:", staged.predict(X_test[:1]))

## 6. Mini-project: compare and justify the winner

Write your comparison below: which run won, on which metric, and whether the difference
is meaningful (try re-running with different `random_state` splits to see variance).
Then promote your winner to the `production` alias with `set_registered_model_alias`.

In [ ]:
summary = runs.sort_values("metrics.accuracy", ascending=False)
summary[["tags.mlflow.runName", "metrics.accuracy"]].head(10)